# Loofah Image Analysis Pipeline
This notebook demonstrates a complete image analysis pipeline using the qim3d library. We'll analyse a loofah sample and carry out the following steps for analysis:
1. **Data Downloading:** Loading the scan from the qim3d repository
2. **Visualization:** Interactive exploration of the 3D structure
3. **Density Analysis:** Quantitative analysis of loofah fibre density

In [14]:
import qim3d
from tifffile import TiffFile
from pprint import pprint
from skimage.filters import threshold_otsu
import numpy as np
import gc
import matplotlib.pyplot as plt

# 1. Downloading and Loading the data
The qim3d data repository provides convenient access to various samples scanned at different resolutions. For computational efficiency during exploration and prototyping, downsampled versions are available alongside full-resolution samples. The `qim3d.io.Downloader` is a simple interface for listing, downloading and loading these samples.

In [15]:
downloader = qim3d.io.Downloader()

We can list the available samples by calling `downloader.list_files()` which shows a list of all downloadable files from the QIM data repository and categorises them. 

In [16]:
downloader.list_files()


╭──────╮
│ Coal │
╰──────╯
Coal.CoalBrikett                                  (2.24GB)
Coal.CoalBrikettZoom_DOWNSAMPLED                  (238.50MB)
Coal.CoalBrikett_Zoom                             (3.73GB)

╭────────╮
│ Corals │
╰────────╯
Corals.Coral_1                                    (2.27GB)
Corals.Coral_2                                    (2.38GB)
Corals.Coral2_DOWNSAMPLED                         (152.66MB)
Corals.Coral_1_1                                  (1.83GB)
Corals.Coral_1_2                                  (1.83GB)
Corals.Coral_1_3                                  (1.83GB)
Corals.MexCoral                                   (2.24GB)

╭─────────────╮
│ Cowry_Shell │
╰─────────────╯
Cowry_Shell.Cowry_DOWNSAMPLED                     (116.91MB)
Cowry_Shell.Cowry_Shell                           (1.83GB)

╭──────╮
│ Crab │
╰──────╯
Crab.HerrmitCrab                                  (2.38GB)
Crab.OkinawaCrab                                  (1.86GB)

╭───────────────╮
│ Deer_Man

For this task, we'll use the downsampled loofah dataset which can be downloaded by using the filename. The `load_file=True` parameter automatically loads the data into memory after downloading.

In [17]:
data = downloader.Loofah.Loofah_DOWNSAMPLED(load_file=True)

File already downloaded:
/home/s215170/Repos/qim3d/docs/notebooks/Loofah/Loofah_DOWNSAMPLED.tif

Loading Loofah_DOWNSAMPLED.tif
Using virtual stack


By default, the column is loaded with the parameter `virtual_stack=True` meaning that it isn't fully read into RAM. Only it's metadata and index structure are loaded up front and portions of the volume are lazily loaded.

# 2. Visualization and Data Exploration

To get a solid understanding of the dataset, we can first read the TIFF metadata using a simple loop to display all tags:
- `BitsPerSample: 16` - The bit depth of each pixel.
- `IJMetadata` - ImageJ-specific metadata containing individual filenames for each of the slices.
- `IJMetadataByteCounts` - size in bytes for each slice.
- `Image Description` - ImageJ parameters:
    - `ImageJ=1.53a` Version of ImageJ used to create the file
    - `images=300` Total number of image slices in the stack
    - `slices=300` Number of Z-slices
    - `loop=false` Animation setting for ImageJ
    - `min=0.0, max=65535.0` Intensity range of the image data
- `ImageLength: 500` - The height of each image slice in pixels (rows).
- `ImageWidth: 500` - The width of each image slice in pixels (columns).
- `NewSubfileType: <FILETYPE.UNDEFINED: 0>` - TIFF tag indicating the type of data in the file (undefined means it's a standard image).
- `PhotometricInterpretation: <PHOTOMETRIC.MINISBLACK: 1>` - How pixel values should be interpreted (MINISBLACK means 0 = black, higher values = lighter).
- `RowsPerStrip: 500` - Number of rows stored in each data strip (matches image height, so each slice is one strip).
- `SamplesPerPixel: 1` - Number of color channels per pixel (1 = grayscale, 3 = RGB).
- `StripByteCounts: (500000,)` - Size in bytes of each data strip (500x500x2 bytes = 500,000 bytes per slice).
- `StripOffsets: (9841,)` - Byte offset where pixel data begins in the file.

In [18]:
with TiffFile("Loofah/Loofah_DOWNSAMPLED.tif") as tif:
    meta = {tag.name: tag.value for tag in tif.pages[0].tags.values()}

pprint(meta)

{'BitsPerSample': 16,
 'IJMetadata': {'Labels': ['Loofah0000.tif',
                           'Loofah0001.tif',
                           'Loofah0002.tif',
                           'Loofah0003.tif',
                           'Loofah0004.tif',
                           'Loofah0005.tif',
                           'Loofah0006.tif',
                           'Loofah0007.tif',
                           'Loofah0008.tif',
                           'Loofah0009.tif',
                           'Loofah0010.tif',
                           'Loofah0011.tif',
                           'Loofah0012.tif',
                           'Loofah0013.tif',
                           'Loofah0014.tif',
                           'Loofah0015.tif',
                           'Loofah0016.tif',
                           'Loofah0017.tif',
                           'Loofah0018.tif',
                           'Loofah0019.tif',
                           'Loofah0020.tif',
                           'Loofa

qim3d offers multiple methods for visualiztaion, including interactive viewers and 3D volumetric rendering. Let's explore the dataset using the `qim3d.viz.slicer_orthogonal()` first. This viewer shows three perpendicular cross-sections (Axial (Z), Coronal (Y), Sagittal (X)), each with its own slider to adjust the slice index in real time.

In [19]:
qim3d.viz.slicer_orthogonal(data, color_map="magma")

The `qim3d.viz.volumetric()` function provides an interactive 3D volumetric renderer that allows you to explore the internal structure of the loofah sample in three dimensions. Unlike the orthogonal slicer which shows 2D cross-sections, this volumetric renderer creates a translucent 3D representation where you can:

- **Rotate and zoom** the volume using mouse controls to examine the structure from different angles
- **Adjust opacity** to see through the outer layers and reveal internal features
- **Control transfer functions** to highlight different density ranges within the material

The volumetric renderer is particularly useful for understanding the complex 3D architecture of porous materials like loofah, where the interconnected fiber network creates intricate pathways and void spaces that are difficult to appreciate from 2D slices alone.

In [20]:
qim3d.viz.volumetric(data)

/home/s215170/miniconda3/envs/qim3d/lib/python3.11/site-packages/traittypes/traittypes.py:97: UserWarning:

Given trait value dtype "float16" does not match required type "float64". A coerced copy has been created.



Output()

The raw volumetric data contains a dark boundary around the loofah sample that appears as low-intensity values (below 20,000). This boundary represents imaging artifacts rather than actual material structure. We can remove these artifacts by setting all pixels below a threshold value to zero, effectively creating a cleaned dataset that focuses on the actual loofah fiber structure. This preprocessing step improves the quality of subsequent analysis and visualization by eliminating noise and artifacts from the imaging process.


In [21]:
cleaned_data = data.copy()
cleaned_data[cleaned_data < 20000] = 0
qim3d.viz.volumetric(cleaned_data)

Output()

# 3. Image Analysis - Density Measurement

Now that we've visualized and cleaned our loofah dataset, we'll perform quantitative analysis to measure key structural properties. In this section, we'll:

1. **Explore thresholding interactively** using qim3d's threshold exploration tool
2. **Apply morphological operations** using qim3d's morphology functions
3. **Visualize each step** of the segmentation pipeline
4. **Calculate fiber density** with proper validation

## 3.1 Interactive Threshold Exploration

Instead of automatically applying Otsu's threshold, let's use qim3d's interactive threshold exploration tool to better understand how different threshold values affect our segmentation:

In [22]:
# Interactive threshold exploration - this helps us understand the optimal threshold
qim3d.viz.threshold(cleaned_data)

In [23]:
# After exploring interactively, let's also check what Otsu suggests
threshold_value = threshold_otsu(cleaned_data)
print(f"Otsu's suggested threshold: {threshold_value}")

# Apply the threshold (you can adjust this value based on the interactive exploration)
binary_mask = cleaned_data > threshold_value

Otsu's suggested threshold: 0


## 3.2 Visualizing the Segmentation Pipeline

Let's visualize each step of our segmentation to understand what we're measuring:

### Step 1: Binary segmentation after thresholding

In [24]:
qim3d.viz.volumetric(binary_mask)

Output()

### Step 2: Apply morphological operations using qim3d

Let's clean up the binary segmentation using qim3d's morphological operations:

In [27]:
# Use qim3d's morphological closing to clean up the segmentation
cleaned_binary = qim3d.morphology.closing(binary_mask, kernel=2, method='scipy.ndimage')

# Visualize the result
qim3d.viz.volumetric(cleaned_binary.astype(np.uint8))

fiber_volume = np.sum(cleaned_binary)
print(f"Threshold value used: {threshold_value}")
print(f"Fiber volume: {fiber_volume:,} voxels")
print(f"This represents {fiber_volume/np.prod(data.shape)*100:.1f}% of the total scan volume")

Output()

Threshold value used: 0
Fiber volume: 2,793,473 voxels
This represents 3.7% of the total scan volume


To calculate the Total Loofah Volume, we use a two-step approach:
1. **Gaussian blur** (`sigma=16`): Smooths the data and fills small gaps
2. **Morphological closing** (`ball(radius=5)`): Fills remaining holes and creates a continuous outer envelope

In [28]:
# Step 1: Create a mask of non-zero regions
print("Creating loofah envelope...")
loofah_mask = cleaned_data > 0

# Step 2: Apply Gaussian blur to smooth and fill gaps
print("Step 1: Applying Gaussian blur to create smooth envelope...")
blurred_mask = qim3d.filters.gaussian(loofah_mask.astype(float), sigma=8)

# Visualize the blurred result
qim3d.viz.volumetric((blurred_mask > 0.1).astype(np.uint8))

# Step 3: Apply morphological closing using qim3d to create solid envelope
print("Step 2: Applying morphological closing to create solid envelope...")
envelope_mask = qim3d.morphology.closing(blurred_mask > 0.1, kernel=10, method='scipy.ndimage')

# Visualize the final envelope
qim3d.viz.volumetric(envelope_mask.astype(np.uint8))

loofah_envelope_volume = np.sum(envelope_mask)
print(f"\nLoofah envelope volume: {loofah_envelope_volume:,} voxels")
print(f"Envelope represents {loofah_envelope_volume/np.prod(data.shape)*100:.1f}% of the total scan volume")

Creating loofah envelope...
Step 1: Applying Gaussian blur to create smooth envelope...


Output()

Step 2: Applying morphological closing to create solid envelope...


Output()


Loofah envelope volume: 14,445,976 voxels
Envelope represents 19.3% of the total scan volume


From this, we can calculate the density of the loofah

In [29]:
# Calculate density
density = fiber_volume / loofah_envelope_volume
print(f"\nFinal Results:")
print(f"Fiber volume: {fiber_volume:,} voxels")
print(f"Loofah envelope volume: {loofah_envelope_volume:,} voxels")
print(f"Calculated density: {density:.3f} ({density*100:.1f}%)")

# Let's also check what percentage of the envelope is actually fiber
envelope_only_volume = loofah_envelope_volume - fiber_volume
print(f"Void space volume: {envelope_only_volume:,} voxels ({envelope_only_volume/loofah_envelope_volume*100:.1f}%)")


Final Results:
Fiber volume: 2,793,473 voxels
Loofah envelope volume: 14,445,976 voxels
Calculated density: 0.193 (19.3%)
Void space volume: 11,652,503 voxels (80.7%)
